convergence.py -- step refinement and error calculations

Question answered: what time step does forward Euler need so that the quantities in the study (peak load, skip / no-skip classification, and the entry-angle limits) are accurate?

  A. State error at a fixed time vs dt          -> Euler is first order (error ~ dt)
  B. Peak aerodynamic load vs dt                -> relative error, observed order, Richardson estimate
  C. Entry-angle boundaries vs dt               -> skip boundary and load-limit boundary angles
  D. Stability estimate along the trajectory    -> largest dt that keeps Euler stable (linearized)
  E. dt recommendation from the tolerances below

Reference solution: RK4 ($dt = 0.1 s$) on model.py equations. RK4 is used ONLY as a
high-accuracy yardstick here, never in the study itself. Check 5 of verification.py already showed
that this reference agrees with an independently coded version of the equations.

Run:   python convergence.py            (a few minutes)
       python convergence.py --quick    (coarser, about 1 minute)
Outputs: results/convergence_*.csv and results/fig_convergence.png  (plots.py can reuse the CSVs)

In [ ]:
import os
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import parameters as P
from verification import (team_rhs, team_load, euler_step, rk4_step, march, tail_slope, table,
                          S0, H0, V0, GAMMA0, V_CAPTURE, T_MAX)

LOAD_LIMIT_G = getattr(P, "LOAD_LIMIT_G", 9.0)     # [g] load threshold used in the study
TOL_PEAK = 0.01                                    # accept <= 1 % error in peak load
TOL_BOUNDARY_DEG = 0.01                            # accept <= 0.01 deg shift in a boundary angle
BRACKET_DEG = (-12.0, -2.0)                        # steep end (no skip) .. shallow end (skip)
REF_DT = 0.1                                       # RK4 reference step [s]
OUT = "results"


# ---------------------------------------------------------------- single entry run with events
def run_entry(gamma0, V0_=V0, dt=0.05, method="euler"):
    """March one entry until a terminal event (same rules as the study).
    skip     : altitude has been below H0 and climbs back through H0
    impact   : h <= 0
    captured : V < V_CAPTURE
    Peak load is the maximum over this first encounter, sampled at every step."""
    if method == "euler":
        step = lambda s: euler_step(s, dt)
    else:
        step = lambda s: rk4_step(team_rhs, s, dt)
    s = np.array([H0, V0_, gamma0, 0.0])
    peak, t_peak, below = team_load(s), 0.0, False
    for k in range(1, int(round(T_MAX / dt)) + 1):
        s = step(s)
        n = team_load(s)
        if n > peak:
            peak, t_peak = n, k * dt
        below |= s[0] < H0
        if below and s[0] >= H0:
            return dict(skip=True, peak=peak, t_peak=t_peak, t_end=k * dt, outcome="skip")
        if s[0] <= 0:
            return dict(skip=False, peak=peak, t_peak=t_peak, t_end=k * dt, outcome="impact")
        if s[1] < V_CAPTURE:
            return dict(skip=False, peak=peak, t_peak=t_peak, t_end=k * dt, outcome="captured")
    return dict(skip=False, peak=peak, t_peak=t_peak, t_end=T_MAX, outcome="timeout")


def bisect_angle(pred, lo_deg, hi_deg, tol_deg=1e-4):
    """pred(gamma_rad) is False at the steep end and True at the shallow end. Returns the flip angle [deg]
    (nan if the bracket does not contain a flip)."""
    f = lambda a: pred(np.deg2rad(a))
    if f(lo_deg) or not f(hi_deg):
        return np.nan
    while hi_deg - lo_deg > tol_deg:
        mid = 0.5 * (lo_deg + hi_deg)
        lo_deg, hi_deg = (lo_deg, mid) if f(mid) else (mid, hi_deg)
    return 0.5 * (lo_deg + hi_deg)


# ---------------------------------------------------------------- A. state error at fixed time
def study_state(dts, T):
    ref = march(lambda s, d: rk4_step(team_rhs, s, d), S0, T, REF_DT)[-1]
    rows = []
    for dt in dts:
        s = march(lambda s_, d: euler_step(s_, d), S0, T, dt)[-1]
        rows.append((dt, *np.abs(s - ref)[:3]))
    rows = np.array(rows)
    table(["dt [s]", "|dh| [m]", "|dV| [m/s]", "|dgamma| [rad]"], rows)
    p = [tail_slope(rows[:, 0], rows[:, j]) for j in (1, 2, 3)]
    print(f"    observed order: h {p[0]:.2f}, V {p[1]:.2f}, gamma {p[2]:.2f}  (forward Euler: expect 1)")
    return rows


# ---------------------------------------------------------------- B. peak load vs dt
def study_peak(dts):
    ref = run_entry(GAMMA0, dt=REF_DT, method="rk4")
    q = np.array([run_entry(GAMMA0, dt=dt)["peak"] for dt in dts])
    err = np.abs(q - ref["peak"]) / ref["peak"]
    # Richardson extrapolation (p = 1) from consecutive step sizes: Q_inf ~ Q_f + (Q_f - Q_c)/(r - 1), r = dt_c/dt_f
    r = dts[:-1] / dts[1:]
    q_rich = q[1:] + (q[1:] - q[:-1]) / (r - 1)
    print(f"    reference (RK4, dt={REF_DT}): peak load = {ref['peak']:.4f} g at t = {ref['t_peak']:.1f} s, outcome = {ref['outcome']}")
    rows = np.column_stack([dts, q, err, np.r_[np.nan, q_rich]])
    table(["dt [s]", "peak load [g]", "rel. error", "Richardson Q_inf"], rows)
    print(f"    observed order of peak-load error: {tail_slope(dts, err):.2f}")
    return rows, ref


# ---------------------------------------------------------------- C. boundary angles vs dt
def study_boundaries(dts):
    def both(dt, method):
        skip = bisect_angle(lambda g: run_entry(g, dt=dt, method=method)["skip"], *BRACKET_DEG)
        load = bisect_angle(lambda g: run_entry(g, dt=dt, method=method)["peak"] <= LOAD_LIMIT_G, *BRACKET_DEG)
        return skip, load
    ref_skip, ref_load = both(REF_DT, "rk4")
    print(f"    reference (RK4, dt={REF_DT}): skip boundary = {ref_skip:.4f} deg, {LOAD_LIMIT_G:g} g boundary = {ref_load:.4f} deg")
    rows = []
    for dt in dts:
        s_b, l_b = both(dt, "euler")
        rows.append((dt, s_b, s_b - ref_skip, l_b, l_b - ref_load))
    rows = np.array(rows)
    table(["dt [s]", "skip bdry [deg]", "shift [deg]", "load bdry [deg]", "shift [deg]"], rows)
    return rows, (ref_skip, ref_load)


# ---------------------------------------------------------------- D. stability estimate
def jacobian(s):
    J = np.zeros((4, 4))
    for j, scale in enumerate([1e3, 1e3, 0.1, 1.0]):
        e = 1e-6 * max(abs(s[j]), scale)
        sp, sm = s.copy(), s.copy()
        sp[j] += e
        sm[j] -= e
        J[:, j] = (team_rhs(sp) - team_rhs(sm)) / (2 * e)
    return J


def study_stability(stride=10):
    """Forward Euler is stable for a mode lambda when |1 + lambda dt| < 1, i.e. dt < -2 Re(lambda)/|lambda|^2.
    Linearize along the reference trajectory and take the most restrictive decaying mode."""
    ref = run_entry(GAMMA0, dt=REF_DT, method="rk4")
    n = int(round(ref["t_end"] / REF_DT))
    hist = march(lambda s, d: rk4_step(team_rhs, s, d), S0, n * REF_DT, REF_DT)
    t, lim = [], []
    for k in range(0, n + 1, stride):
        lam = np.linalg.eigvals(jacobian(hist[k]))
        lam = lam[lam.real < -1e-12]
        t.append(k * REF_DT)
        lim.append(np.min(-2 * lam.real / np.abs(lam) ** 2) if lam.size else np.inf)
    t, lim = np.array(t), np.array(lim)
    i = int(np.argmin(lim))
    print(f"    smallest stable dt along the baseline trajectory: {lim[i]:.3g} s (at t = {t[i]:.0f} s)")
    return np.column_stack([t, lim]), lim[i]


# ---------------------------------------------------------------- E. recommendation
def recommend(peak_rows, bdry_rows, dt_stab):
    ok = (peak_rows[:, 2] <= TOL_PEAK) & (np.abs(bdry_rows[:, 2]) <= TOL_BOUNDARY_DEG) \
         & (np.abs(bdry_rows[:, 4]) <= TOL_BOUNDARY_DEG)
    dts = peak_rows[:, 0]
    order = np.argsort(dts)[::-1]                    # coarse -> fine
    good = None
    for i in order:                                  # largest dt such that it AND every finer dt pass
        if all(ok[dts <= dts[i]]):
            good = dts[i]
            break
    print(f"\nE. Tolerances: peak load <= {TOL_PEAK*100:g} %, boundary angles <= {TOL_BOUNDARY_DEG:g} deg")
    if good is None:
        print("   No tested dt meets the tolerances -- extend the dt list to smaller steps.")
    else:
        print(f"   Recommended dt = {good:g} s  ({int(round(100.0 / good))} steps per 100 s of flight; "
              f"{good / dt_stab * 100:.1f} % of the linear stability limit {dt_stab:.3g} s)")
    return good


def save(name, rows, header):
    os.makedirs(OUT, exist_ok=True)
    np.savetxt(os.path.join(OUT, name), rows, delimiter=",", header=header, comments="")


def make_figure(state, peak, bdry, stab, dt_rec, T):
    fig, ax = plt.subplots(2, 2, figsize=(11, 8))
    a = ax[0, 0]
    for j, lab in zip((1, 2, 3), ("$|\\Delta h|$ [m]", "$|\\Delta V|$ [m/s]", "$|\\Delta\\gamma|$ [rad]")):
        a.loglog(state[:, 0], state[:, j], "o-", label=lab)
    a.loglog(state[:, 0], state[0, 2] * state[:, 0] / state[0, 0], "k--", lw=0.8, label="slope 1")
    a.set(xlabel="dt [s]", ylabel="error vs RK4 reference", title=f"A. State error at t = {T:.1f} s")
    a.legend(fontsize=8)
    a = ax[0, 1]
    a.loglog(peak[:, 0], peak[:, 2], "o-", label="peak-load error")
    a.loglog(peak[:, 0], peak[0, 2] * peak[:, 0] / peak[0, 0], "k--", lw=0.8, label="slope 1")
    a.axhline(TOL_PEAK, color="r", ls=":", label="tolerance")
    a.set(xlabel="dt [s]", ylabel="relative error", title="B. Peak aerodynamic load")
    a.legend(fontsize=8)
    a = ax[1, 0]
    a.loglog(bdry[:, 0], np.abs(bdry[:, 2]), "o-", label="skip boundary")
    a.loglog(bdry[:, 0], np.abs(bdry[:, 4]), "s-", label=f"{LOAD_LIMIT_G:g} g boundary")
    a.axhline(TOL_BOUNDARY_DEG, color="r", ls=":", label="tolerance")
    a.set(xlabel="dt [s]", ylabel="|angle shift| [deg]", title="C. Entry-angle boundaries")
    a.legend(fontsize=8)
    a = ax[1, 1]
    a.semilogy(stab[:, 0], stab[:, 1], "-")
    if dt_rec:
        a.axhline(dt_rec, color="r", ls=":", label=f"recommended dt = {dt_rec:g} s")
        a.legend(fontsize=8)
    a.set(xlabel="time since entry [s]", ylabel="max stable dt [s]", title="D. Linearized Euler stability limit")
    for x in ax.ravel():
        x.grid(alpha=0.3, which="both")
    fig.tight_layout()
    os.makedirs(OUT, exist_ok=True)
    fig.savefig(os.path.join(OUT, "fig_convergence.png"), dpi=170)


def main(quick=False):
    dts = np.array([0.4, 0.1, 0.025]) if quick else np.array([0.4, 0.2, 0.1, 0.05, 0.025, 0.0125])
    dts_state = np.array([1.6, 0.4, 0.1, 0.025]) if quick else np.array([1.6, 0.8, 0.4, 0.2, 0.1, 0.05, 0.025])
    print(f"convergence.py -- baseline: h0 = {H0/1e3:.2f} km, V0 = {V0/1e3:.3f} km/s, gamma0 = {np.rad2deg(GAMMA0):.2f} deg")

    base = run_entry(GAMMA0, dt=0.05)
    T = 1.6 * max(2, round(min(2 * base["t_peak"], base["t_end"]) / 1.6))     # fixed comparison time, covers the peak
    print(f"\nbaseline (Euler, dt=0.05): outcome = {base['outcome']}, peak load {base['peak']:.2f} g at t = {base['t_peak']:.1f} s")

    print(f"\nA. State error at t = {T:.1f} s")
    state = study_state(dts_state, T)
    print("\nB. Peak aerodynamic load, baseline entry angle")
    peak, _ = study_peak(dts)
    print("\nC. Entry-angle boundaries (bisection, 1e-4 deg)")
    bdry, refs = study_boundaries(dts)
    print("\nD. Stability")
    stab, dt_stab = study_stability()
    dt_rec = recommend(peak, bdry, dt_stab)

    save("convergence_state.csv", state, "dt_s,abs_dh_m,abs_dV_mps,abs_dgamma_rad")
    save("convergence_peak.csv", peak, "dt_s,peak_load_g,rel_error,richardson_estimate_g")
    save("convergence_boundaries.csv", bdry, "dt_s,skip_boundary_deg,skip_shift_deg,load_boundary_deg,load_shift_deg")
    save("convergence_stability.csv", stab, "t_s,max_stable_dt_s")
    make_figure(state, peak, bdry, stab, dt_rec, T)
    print(f"\nsaved CSVs and fig_convergence.png in ./{OUT}/")
    return dict(state=state, peak=peak, boundaries=bdry, stability=stab, dt_recommended=dt_rec)


if __name__ == "__main__":
    main(quick="--quick" in sys.argv)